In [ ]:
import os
import pickle

import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 1. Read in the data

In [ ]:
# read in pickle file
model_name = "chronos-t5-base"
config_key = "rf2_sl8"

results_dir = "/work/results/rrt_induction_scores"
with open(os.path.join(results_dir, model_name, config_key, "center_scores.pkl"), "rb") as f:
    center_scores = pickle.load(f)
    center_scores_mean, center_scores_std = center_scores["mean"], center_scores["std"]

with open(os.path.join(results_dir, model_name, config_key, "right_scores.pkl"), "rb") as f:
    right_scores = pickle.load(f)
    right_scores_mean, right_scores_std = right_scores["mean"], right_scores["std"]

with open(
    os.path.join(results_dir, model_name, config_key, "expected_token_attribution.pkl"),
    "rb",
) as f:
    expected_token_attribution = pickle.load(f)
    expected_token_attribution_mean, expected_token_attribution_std = (
        expected_token_attribution["mean"],
        expected_token_attribution["std"],
    )

with open(os.path.join(results_dir, model_name, config_key, "rrt_vars.pkl"), "rb") as f:
    rrt_vars = pickle.load(f)
    std_factor, overall_scores_mean, overall_scores_std = (
        rrt_vars["std_factor"],
        rrt_vars["overall_scores"][0],
        rrt_vars["overall_scores"][1],
    )


## 2. Create Induction Mosaic

In [ ]:
# Create a 1x2 subplot with heatmaps
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Center Scores", "Right Scores"),
    horizontal_spacing=0.1,
)

# Add heatmap for center scores
fig.add_trace(
    go.Heatmap(
        z=center_scores_mean,
        colorscale="Viridis",
        colorbar=dict(title="Attention", x=0.46),
        hovertemplate="Layer: %{y}<br>Head: %{x}<br>Score: %{z:.3f}<extra></extra>",
    ),
    row=1,
    col=1,
)

# Add heatmap for right scores
fig.add_trace(
    go.Heatmap(
        z=right_scores_mean,
        colorscale="Viridis",
        colorbar=dict(title="Attention", x=1.0),
        hovertemplate="Layer: %{y}<br>Head: %{x}<br>Score: %{z:.3f}<extra></extra>",
    ),
    row=1,
    col=2,
)

# Update layout
fig.update_layout(
    height=600,
    width=1000,
    title_text="Induction Attention Patterns",
    yaxis_title="Decoder Layer",
    xaxis_title="Attention Head",
    xaxis2_title="Attention Head",
)

# Update axes to show integer indices
fig.update_xaxes(tickmode="linear", dtick=1)
fig.update_yaxes(tickmode="linear", dtick=1)

# Show the figure
fig.show()


## Plot expected token attributions

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create labels and organize data
labels = []
right_means = []
right_stds = []
attribution_means = []
attribution_stds = []

for layer in range(right_scores_mean.shape[0]):
    for head in range(right_scores_mean.shape[1]):
        labels.append(f"L{layer}H{head}")
        right_means.append(right_scores_mean[layer, head])
        right_stds.append(right_scores_std[layer, head])
        attribution_means.append(expected_token_attribution_mean[layer, head])
        attribution_stds.append(expected_token_attribution_std[layer, head])

# Create indices for different sorting methods
indices_by_layer = list(range(len(labels)))  # Default order is by layer
indices_by_induction = np.argsort(right_means)[::-1]  # Sort by induction score (descending)
indices_by_attribution = np.argsort(attribution_means)[::-1]  # Sort by attribution score (descending)

# Create figure with two subplots (one below the other)
fig = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=(
        "Induction Scores",
        "Expected Token Attribution",
    ),
    vertical_spacing=0.15,
)

# Create traces for each sorting method
# By layer
fig.add_trace(
    go.Bar(
        x=[labels[i] for i in indices_by_layer],
        y=[right_means[i] for i in indices_by_layer],
        error_y=dict(
            type="data",
            array=[right_stds[i] for i in indices_by_layer],
            visible=True,
            thickness=0.7,
            width=4,
            color="rgba(0,0,0,0.3)",
        ),
        marker_color="cornflowerblue",
        width=0.8,
        name="Right Scores",
        visible=True,
    ),
    row=1,
    col=1,
)

# By induction score
fig.add_trace(
    go.Bar(
        x=[labels[i] for i in indices_by_induction],
        y=[right_means[i] for i in indices_by_induction],
        error_y=dict(
            type="data",
            array=[right_stds[i] for i in indices_by_induction],
            visible=True,
            thickness=0.7,
            width=4,
            color="rgba(0,0,0,0.3)",
        ),
        marker_color="cornflowerblue",
        width=0.8,
        name="Right Scores",
        visible=False,
    ),
    row=1,
    col=1,
)

# By attribution score
fig.add_trace(
    go.Bar(
        x=[labels[i] for i in indices_by_attribution],
        y=[right_means[i] for i in indices_by_attribution],
        error_y=dict(
            type="data",
            array=[right_stds[i] for i in indices_by_attribution],
            visible=True,
            thickness=0.7,
            width=4,
            color="rgba(0,0,0,0.3)",
        ),
        marker_color="cornflowerblue",
        width=0.8,
        name="Right Scores",
        visible=False,
    ),
    row=1,
    col=1,
)

# Add bar chart for expected token attribution in the bottom subplot
# By layer
fig.add_trace(
    go.Bar(
        x=[labels[i] for i in indices_by_layer],
        y=[attribution_means[i] for i in indices_by_layer],
        error_y=dict(
            type="data",
            array=[attribution_stds[i] for i in indices_by_layer],
            visible=True,
            thickness=0.7,
            width=4,
            color="rgba(0,0,0,0.3)",
        ),
        marker_color="firebrick",
        width=0.8,
        name="Expected Token Attribution",
        visible=True,
    ),
    row=2,
    col=1,
)

# By induction score
fig.add_trace(
    go.Bar(
        x=[labels[i] for i in indices_by_induction],
        y=[attribution_means[i] for i in indices_by_induction],
        error_y=dict(
            type="data",
            array=[attribution_stds[i] for i in indices_by_induction],
            visible=True,
            thickness=0.7,
            width=4,
            color="rgba(0,0,0,0.3)",
        ),
        marker_color="firebrick",
        width=0.8,
        name="Expected Token Attribution",
        visible=False,
    ),
    row=2,
    col=1,
)

# By attribution score
fig.add_trace(
    go.Bar(
        x=[labels[i] for i in indices_by_attribution],
        y=[attribution_means[i] for i in indices_by_attribution],
        error_y=dict(
            type="data",
            array=[attribution_stds[i] for i in indices_by_attribution],
            visible=True,
            thickness=0.7,
            width=4,
            color="rgba(0,0,0,0.3)",
        ),
        marker_color="firebrick",
        width=0.8,
        name="Expected Token Attribution",
        visible=False,
    ),
    row=2,
    col=1,
)

# Add horizontal line for overall score mean in the expected token attribution graph
# By layer
fig.add_trace(
    go.Scatter(
        x=[labels[indices_by_layer[0]], labels[indices_by_layer[-1]]],
        y=[overall_scores_mean, overall_scores_mean],
        mode="lines",
        line=dict(color="green", width=2, dash="dash"),
        name="Overall Score Mean",
        visible=True,
    ),
    row=2,
    col=1,
)

# By induction score
fig.add_trace(
    go.Scatter(
        x=[labels[indices_by_induction[0]], labels[indices_by_induction[-1]]],
        y=[overall_scores_mean, overall_scores_mean],
        mode="lines",
        line=dict(color="green", width=2, dash="dash"),
        name="Overall Score Mean",
        visible=False,
    ),
    row=2,
    col=1,
)

# By attribution score
fig.add_trace(
    go.Scatter(
        x=[labels[indices_by_attribution[0]], labels[indices_by_attribution[-1]]],
        y=[overall_scores_mean, overall_scores_mean],
        mode="lines",
        line=dict(color="green", width=2, dash="dash"),
        name="Overall Score Mean",
        visible=False,
    ),
    row=2,
    col=1,
)

# Create dropdown menu
updatemenus = [
    dict(
        active=0,
        buttons=[
            dict(
                label="Sort by Layer",
                method="update",
                args=[
                    {
                        "visible": [
                            True,
                            False,
                            False,
                            True,
                            False,
                            False,
                            True,
                            False,
                            False,
                        ]
                    },
                    {"title": "Expected Token Attribution"},
                ],
            ),
            dict(
                label="Sort by Induction Score",
                method="update",
                args=[
                    {
                        "visible": [
                            False,
                            True,
                            False,
                            False,
                            True,
                            False,
                            False,
                            True,
                            False,
                        ]
                    },
                    {"title": "Expected Token Attribution"},
                ],
            ),
            dict(
                label="Sort by Attribution Score",
                method="update",
                args=[
                    {
                        "visible": [
                            False,
                            False,
                            True,
                            False,
                            False,
                            True,
                            False,
                            False,
                            True,
                        ]
                    },
                    {"title": "Expected Token Attribution"},
                ],
            ),
        ],
        direction="down",
        pad={"r": 10, "t": 10},
        showactive=True,
        x=0.8,
        xanchor="left",
        y=1.15,
        yanchor="top",
    ),
]

# Update layout
fig.update_layout(
    title="Expected Token Attribution by Layer",
    updatemenus=updatemenus,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    height=800,
    width=1000,
    bargap=0.15,
)

# Set y-axes titles
fig.update_yaxes(title_text="Mean Right Score", row=1, col=1)
fig.update_yaxes(title_text="Mean Expected Token Attribution", row=2, col=1)
fig.update_xaxes(title_text="Attention Head", row=2, col=1)

# Show the figure
fig.show()